# ETAP AI Platform - Agent Evaluation

This notebook evaluates the quality of the power-system engineering agents using the LangWatch Evaluations API.

In [ ]:
import os
import pandas as pd
from langwatch import LangWatch

# Initialize LangWatch client
langwatch = LangWatch(api_key=os.environ.get("LANGWATCH_API_KEY"))

# Load evaluation dataset
eval_data = pd.DataFrame([
    {
        "input": "Run a load flow study on a 5-bus industrial system with 2 generators and 3 loads.",
        "expected_output": "The agent should request system parameters, explain Newton-Raphson convergence, and present voltage profiles.",
        "agent": "load-flow-agent"
    },
    {
        "input": "Calculate the arc flash incident energy at a 480V switchboard with 35kA bolted fault current.",
        "expected_output": "The agent should cite IEEE 1584-2018, compute incident energy, and recommend PPE level.",
        "agent": "arcflash-agent"
    },
    {
        "input": "Verify protection coordination for a 13.8kV feeder with 1200A upstream relay and 600A downstream relay.",
        "expected_output": "The agent should reference IEC 60255, calculate coordination margins, and confirm selectivity.",
        "agent": "protection-agent"
    },
])

print(f"Loaded {len(eval_data)} evaluation cases")
eval_data.head()

## Evaluation 1: Engineering Accuracy

Compare agent outputs against expected engineering standards using an LLM-as-judge.

In [ ]:

def evaluate_accuracy(row: pd.Series) -> dict:
    """
    Run an LLM-based evaluation to compare the agent output against the expected output.
    Returns a score from 0 to 1 and reasoning.
    """
    # In production, this would call the actual agent and then compare
    # For the notebook template, we raise an error to prevent accidental reporting of fake scores.
    raise NotImplementedError(
        "This evaluation is a template. Wire in the actual agent call and LLM-as-judge logic "
        "before running in production."
    )
    
eval_data[["accuracy_score", "accuracy_reasoning"]] = eval_data.apply(
    lambda row: pd.Series(evaluate_accuracy(row)), axis=1
)

print(f"Average accuracy score: {eval_data['accuracy_score'].mean():.2f}")
eval_data[["agent", "accuracy_score", "accuracy_reasoning"]]

## Evaluation 2: Standards Compliance

Check that the agent references the correct IEEE/IEC standards.

In [ ]:
STANDARDS_MAP = {
    "load-flow-agent": ["IEEE 3001.2", "IEC 60909"],
    "arcflash-agent": ["IEEE 1584-2018"],
    "protection-agent": ["IEC 60255", "IEEE C37.90"],
    "short-circuit-agent": ["IEC 60909-0:2016"],
    "motorstarting-agent": ["IEEE 3001.2"],
}

def check_standards(row: pd.Series) -> dict:
    STANDARDS_MAP.get(row['agent'], [])
    # In production: analyze agent output for standard mentions
    raise NotImplementedError(
        "This evaluation is a template. Wire in the actual agent output analysis "
        "before running in production."
    )

eval_data[["standards_expected", "standards_mentioned", "compliance_score"]] = eval_data.apply(
    lambda row: pd.Series(check_standards(row)), axis=1
)

print(f"Average compliance score: {eval_data['compliance_score'].mean():.2f}")
eval_data[["agent", "compliance_score", "standards_expected"]]

## Evaluation 3: Safety & Security

Verify that the agent does not hallucinate numerical results and asks for missing data.

In [ ]:
def evaluate_safety(row: pd.Series) -> dict:
    """
    Check that the agent does not fabricate numerical results without input data.
    """
    # In production: analyze agent output for invented numbers
    raise NotImplementedError(
        "This evaluation is a template. Wire in the actual agent output analysis "
        "before running in production."
    )

eval_data[["safety_score", "safety_reasoning"]] = eval_data.apply(
    lambda row: pd.Series(evaluate_safety(row)), axis=1
)

print(f"Average safety score: {eval_data['safety_score'].mean():.2f}")
eval_data[["agent", "safety_score"]]

## Summary Report

Aggregate scores and identify gaps.

In [ ]:
summary = eval_data.groupby('agent').agg({
    'accuracy_score': 'mean',
    'compliance_score': 'mean',
    'safety_score': 'mean'
}).reset_index()

summary['overall'] = (
    summary['accuracy_score'] * 0.5 +
    summary['compliance_score'] * 0.3 +
    summary['safety_score'] * 0.2
)

print("=== Evaluation Summary ===")
print(summary.to_string(index=False))

# Send to LangWatch (optional)
# langwatch.evaluations.create(
#     name="etap-agent-evaluation",
#     dataset=eval_data.to_dict('records'),
#     results=summary.to_dict('records')
# )